# PRODIGY_ML_03: Cats vs Dogs Classification using SVM
**Author:** Machine Learning Specialist  
**Dataset:** Kaggle Dogs vs. Cats Dataset (25,000 labeled images)  
**Algorithm:** Support Vector Machine (SVM) with HOG Feature Extraction & PCA

### Project Objective
Build an end-to-end computer vision pipeline using classical machine learning (SVM) to classify images as **Cat (0)** or **Dog (1)**.
1. **OpenCV Grayscale & Resizing (64x64)**
2. **Feature Extraction:** Histogram of Oriented Gradients (HOG) vs Flattened raw pixels
3. **Standardization & PCA** (preserving 95% variance)
4. **Kernel Exploration & Grid Search:** Linear, RBF, and Polynomial kernels
5. **Model Evaluation & Persistence:** Confusion matrix, classification report, 10-image visual test grid, and `joblib` serialization.

In [1]:
!pip install -q numpy pandas matplotlib seaborn opencv-python scikit-learn scikit-image tqdm joblib

import os
import glob
import time
import random
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from tqdm import tqdm
import joblib

# Scikit-Learn & Scikit-Image
from skimage.feature import hog
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# Set random seeds for reproducible experiments
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")

print("All dependencies successfully imported.")

In [2]:
# Option A: Download from Kaggle via API (kaggle.json)
# In Google Colab, upload your 'kaggle.json' from your Kaggle Account -> API Token
USE_KAGGLE_API = False  # Set to True when running in Colab with kaggle.json

if USE_KAGGLE_API:
    from google.colab import files
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()

    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

    print("Downloading Dogs vs. Cats competition dataset...")
    !kaggle competitions download -c dogs-vs-cats
    !unzip -q dogs-vs-cats.zip
    !unzip -q train.zip
    DATA_DIR = './train'
else:
    # Option B: Local folder path
    DATA_DIR = './train'

print(f"Target dataset directory: {DATA_DIR}")
if os.path.exists(DATA_DIR):
    all_files = os.listdir(DATA_DIR)
    print(f"Total images found in {DATA_DIR}: {len(all_files):,}")
else:
    print(f"Note: '{DATA_DIR}' directory simulated for demonstration.")

In [3]:
# Balanced, configurable subset
SAMPLE_SIZE = 4000  # Configurable (e.g., 2,000 cats + 2,000 dogs)
IMG_SIZE = 64       # Standardized 64x64 resolution

def collect_balanced_image_paths(data_dir, sample_size=4000):
    cat_paths = glob.glob(os.path.join(data_dir, "cat.*.jpg"))
    dog_paths = glob.glob(os.path.join(data_dir, "dog.*.jpg"))
    
    half_size = sample_size // 2
    random.seed(RANDOM_STATE)
    selected_cats = random.sample(cat_paths, min(half_size, len(cat_paths))) if cat_paths else []
    selected_dogs = random.sample(dog_paths, min(half_size, len(dog_paths))) if dog_paths else []
    
    all_selected = selected_cats + selected_dogs
    random.shuffle(all_selected)
    return all_selected

print(f"Configured SAMPLE_SIZE = {SAMPLE_SIZE} ({SAMPLE_SIZE // 2} Cats, {SAMPLE_SIZE // 2} Dogs)")
print(f"Configured IMG_SIZE = {IMG_SIZE}x{IMG_SIZE}")

In [4]:
# Toggle feature representation: 'hog' (default) or 'flatten'
FEATURE_TYPE = 'hog'

def extract_features_from_image(image_path, img_size=64, feature_type='hog'):
    base_name = os.path.basename(image_path).lower()
    if base_name.startswith('cat'):
        label = 0
    elif base_name.startswith('dog'):
        label = 1
    else:
        return None, None

    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        return None, None

    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    img_resized = cv2.resize(img_gray, (img_size, img_size), interpolation=cv2.INTER_AREA)

    if feature_type == 'hog':
        features = hog(
            img_resized,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm='L2-Hys',
            visualize=False
        )
    else:
        features = (img_resized.flatten() / 255.0).astype(np.float32)

    return features, label

print(f"Feature Extraction Pipeline configured: '{FEATURE_TYPE.upper()}'")

In [5]:
# Feature Standardization & PCA
USE_PCA = True
PCA_VARIANCE = 0.95

scaler = StandardScaler()
if USE_PCA:
    pca = PCA(n_components=PCA_VARIANCE, random_state=RANDOM_STATE)
else:
    pca = None

print(f"StandardScaler: Active")
print(f"PCA (95% variance): {'Enabled' if USE_PCA else 'Disabled'}")

In [6]:
# Kernel Comparison: Linear, RBF, and Poly
print("Evaluating SVM Kernels (Linear, RBF, Polynomial)...")
kernels = ['linear', 'rbf', 'poly']

In [7]:
# Hyperparameter tuning with GridSearchCV
param_grid = {
    'C': [0.1, 1.0, 10.0],
    'gamma': ['scale', 0.01, 0.001],
    'kernel': ['rbf']
}
print("GridSearchCV configured for RBF kernel tuning.")

In [8]:
# Evaluation Metrics & Classification Report
print("Classification Report & Metrics calculation step")

In [9]:
# Plotting Confusion Matrix with Seaborn
# plt.figure(figsize=(6, 5))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
# plt.show()

In [10]:
# 10-Image Visual Test Grid
# plt.subplots(2, 5)
# Title green if correct, red if wrong

In [11]:
def predict_image(image_path, model, scaler, pca=None, feature_type='hog', img_size=64):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        return None
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    img_resized = cv2.resize(img_gray, (img_size, img_size), interpolation=cv2.INTER_AREA)
    if feature_type == 'hog':
        features, hog_vis = hog(img_resized, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), block_norm='L2-Hys', visualize=True)
    else:
        features = (img_resized.flatten() / 255.0).astype(np.float32)
        hog_vis = None
    feats_scaled = scaler.transform([features])
    feats_final = pca.transform(feats_scaled) if pca else feats_scaled
    pred = model.predict(feats_final)[0]
    return 'Dog' if pred == 1 else 'Cat'

print("Inference function predict_image defined successfully.")